# Maryland crash costs

Import half-year Maryland CRASH exports into pandas: **Reports** (crash-level), **Vehicles**, **Occupants**, and **NonMotorists**, each combined into one 2024–2025 file per folder.

In [6]:
from pathlib import Path

import pandas as pd

# Run with working directory = this folder (`crash_costs`).
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data"

# Half-year export suffixes in chronological order (H1/H2 2024, then H1/H2 2025).
PERIOD_SUFFIXES = [
    "010124_063124",
    "070124_123124",
    "010125_063125",
    "070125_123125",
]


def combine_period_csvs(
    folder: Path,
    file_prefix: str,
    output_csv: Path,
    *,
    parse_dates: list[str] | None = None,
) -> pd.DataFrame:
    paths = [folder / f"{file_prefix}_{suffix}.csv" for suffix in PERIOD_SUFFIXES]
    read_kw: dict = {"low_memory": False}
    if parse_dates is not None:
        read_kw["parse_dates"] = parse_dates
    frames = [pd.read_csv(p, **read_kw) for p in paths]
    out = pd.concat(frames, ignore_index=True)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(output_csv, index=False)
    return out


reports = combine_period_csvs(
    DATA_DIR / "Reports",
    "Reports",
    DATA_DIR / "Reports" / "reports_maryland_2024_2025.csv",
    parse_dates=["Crashdate", "LOD Crash Date"],
)
occupants = combine_period_csvs(
    DATA_DIR / "Occupants",
    "Occupants",
    DATA_DIR / "Occupants" / "occupants_maryland_2024_2025.csv",
)
nonmotorists = combine_period_csvs(
    DATA_DIR / "NonMotorists",
    "NonMotorists",
    DATA_DIR / "NonMotorists" / "nonmotorists_maryland_2024_2025.csv",
)
vehicles = combine_period_csvs(
    DATA_DIR / "Vehicles",
    "Vehicles",
    DATA_DIR / "Vehicles" / "vehicles_maryland_2024_2025.csv",
)

(
    ("reports", reports.shape),
    ("vehicles", vehicles.shape),
    ("occupants", occupants.shape),
    ("nonmotorists", nonmotorists.shape),
)

(('reports', (210860, 72)),
 ('vehicles', (294295, 69)),
 ('occupants', (470573, 48)),
 ('nonmotorists', (8827, 27)))

In [7]:
# Crash-level table for cost evaluation: key report fields + per-crash injury-status counts
# from occupants (`InjuryStatus Occ`) and nonmotorists (`InjuryStatus NM`), plus vehicle count
# per report (`ReportNumber Veh`), all joined on report id.

import numpy as np

REPORT_ID = "Reportnumber"
VEH_REPORT = "ReportNumber Veh"
OCC_REPORT = "ReportNumber Occ"
OCC_INJURY = "InjuryStatus Occ"
NM_REPORT = "ReportNumber NM"
NM_INJURY = "InjuryStatus NM"

# Sentinel for missing *InjuryStatus* only (distinct from numeric codes like 1–5).
INJURY_STATUS_NA_LABEL = "missing"

REPORT_COLS_FOR_EVAL = [
    "Reportnumber",
    "Crash County Description",
    "Crash Severity Description",
    "Crash Severity",
    "Crashdate",
    "LOD Crash Date",
    "CrashYear",
    "Crashhour",
    "Agencyname",
    "ImpairedCrash",
    "Motorcycle Crash",
    "Non-Motorist Crash",
    "Large Vehicle Involved",
    "Latitude",
    "Longitude",
]


def normalize_report_id(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip()


def injury_status_label(value) -> str:
    if pd.isna(value):
        return INJURY_STATUS_NA_LABEL
    try:
        f = float(value)
        if f.is_integer():
            return str(int(f))
    except (TypeError, ValueError):
        pass
    return str(value)


def injury_status_counts_by_report(
    df: pd.DataFrame,
    report_col: str,
    injury_col: str,
    role_prefix: str,
) -> pd.DataFrame:
    """One row per crash id; columns are counts of persons by injury-status label."""
    t = df[[report_col, injury_col]].copy()
    t[report_col] = normalize_report_id(t[report_col])
    t["_injury_status_label"] = t[injury_col].map(injury_status_label)
    ct = pd.crosstab(t[report_col], t["_injury_status_label"])
    inj_col_safe = injury_col.replace(" ", "_")
    ct.columns = [
        f"{role_prefix}_{inj_col_safe}_{col}_count" for col in ct.columns
    ]
    return ct.reset_index().rename(columns={report_col: REPORT_ID})


base = reports[REPORT_COLS_FOR_EVAL].copy()
base[REPORT_ID] = normalize_report_id(base[REPORT_ID])

occ_counts = injury_status_counts_by_report(
    occupants, OCC_REPORT, OCC_INJURY, "occupants"
)
nm_counts = injury_status_counts_by_report(
    nonmotorists, NM_REPORT, NM_INJURY, "nonmotorists"
)

occ_records = (
    occupants.assign(_rid=normalize_report_id(occupants[OCC_REPORT]))
    .groupby("_rid", observed=False)
    .size()
    .rename("n_occupant_records")
    .reset_index()
    .rename(columns={"_rid": REPORT_ID})
)
nm_records = (
    nonmotorists.assign(_rid=normalize_report_id(nonmotorists[NM_REPORT]))
    .groupby("_rid", observed=False)
    .size()
    .rename("n_nonmotorist_records")
    .reset_index()
    .rename(columns={"_rid": REPORT_ID})
)
veh_records = (
    vehicles.assign(_rid=normalize_report_id(vehicles[VEH_REPORT]))
    .groupby("_rid", observed=False)
    .size()
    .rename("n_vehicle_records")
    .reset_index()
    .rename(columns={"_rid": REPORT_ID})
)

crash_cost_eval = (
    base.merge(occ_counts, on=REPORT_ID, how="left")
    .merge(nm_counts, on=REPORT_ID, how="left")
    .merge(occ_records, on=REPORT_ID, how="left")
    .merge(nm_records, on=REPORT_ID, how="left")
    .merge(veh_records, on=REPORT_ID, how="left")
)
crash_cost_eval["crash_date"] = pd.to_datetime(
    crash_cost_eval["LOD Crash Date"], errors="coerce"
)

injury_count_cols = [c for c in crash_cost_eval.columns if c.endswith("_count")]
crash_cost_eval[injury_count_cols] = (
    crash_cost_eval[injury_count_cols].fillna(0).astype(np.int64)
)
for _col in ("n_occupant_records", "n_nonmotorist_records", "n_vehicle_records"):
    crash_cost_eval[_col] = crash_cost_eval[_col].fillna(0).astype(np.int64)

crash_cost_eval.shape

(210860, 30)

In [8]:
import numpy as np

# Table 1-10 (`police_unit_costs_comp`): apply each cost component row separately (skip Subtotal rows).
# Property-damage crashes (`Crash Severity` == 3): only PDOVehicle unit costs × `n_vehicle_records`
# (no per-person injury costs). All other severities: per-person costs via MAIS/Fatal columns only.

injury_map = {
    1: "Fatal",
    2: "MAIS4",
    3: "MAIS2",
    4: "MAIS1",
    5: "MAIS0",
}

TABLES_XLSX = NOTEBOOK_DIR / "tables_from_report.xlsx"
PDO_SEVERITY_CODE = 3  # Property damage only (Reports export)


def load_police_unit_cost_components(xlsx_path: Path) -> tuple[pd.DataFrame, list[str]]:
    """Rows = cost component labels (excluding blank and 'Subtotal'); cols = PDOVehicle … Fatal."""
    raw = pd.read_excel(xlsx_path, sheet_name="police_unit_costs_comp", header=None)
    categories = [raw.iloc[3, j] for j in range(3, 11)]  # PDOVehicle … Fatal (8 columns)
    categories = [str(c).strip() for c in categories]
    records: list[dict] = []
    component_names: list[str] = []
    for i in range(4, len(raw)):
        label = raw.iloc[i, 2]
        if pd.isna(label):
            continue
        label_s = str(label).strip()
        if label_s == "Subtotal":
            continue
        row_vals = {categories[j]: float(raw.iloc[i, 3 + j]) for j in range(8)}
        records.append(row_vals)
        component_names.append(label_s)
    matrix = pd.DataFrame.from_records(records, index=component_names)
    return matrix, component_names


UNIT_COST_MATRIX, _UNIT_COST_ROW_ORDER = load_police_unit_cost_components(TABLES_XLSX)

# NHTSA table is 2019 $; scale unit costs to June 2025 $ (BLS cumulative inflation, Jun 2019 → Jun 2025).
INFLATION_2019_TO_JUN2025 = 1.26
UNIT_COST_MATRIX = UNIT_COST_MATRIX * INFLATION_2019_TO_JUN2025


def component_to_column_name(component_label: str) -> str:
    safe = (
        component_label.strip()
        .replace(".", "")
        .replace(" ", "_")
        .replace("/", "_")
    )
    return f"estimated_{safe}_comp_cost_2025usd"


def estimated_crash_cost_for_component_row(
    df: pd.DataFrame,
    units: pd.Series,
    *,
    is_pdo: pd.Series,
) -> pd.Series:
    """Per crash: PD → PDOVehicle × n_vehicles; else → Σ person counts × MAIS/Fatal cells (no PDO)."""
    vehicle_cost = df["n_vehicle_records"].astype(float) * float(units["PDOVehicle"])
    person_cost = pd.Series(0.0, index=df.index, dtype="float64")
    for code in ("1", "2", "3", "4", "5"):
        cat = injury_map[int(code)]
        price = float(units[cat])
        occ_col = f"occupants_InjuryStatus_Occ_{code}_count"
        nm_col = f"nonmotorists_InjuryStatus_NM_{code}_count"
        if occ_col in df.columns:
            person_cost = person_cost + df[occ_col].astype(float) * price
        if nm_col in df.columns:
            person_cost = person_cost + df[nm_col].astype(float) * price
    return pd.Series(
        np.where(is_pdo.to_numpy(), vehicle_cost.to_numpy(), person_cost.to_numpy()),
        index=df.index,
        dtype="float64",
    )


is_pdo_crash = crash_cost_eval["Crash Severity"] == PDO_SEVERITY_CODE

for _component, _units in UNIT_COST_MATRIX.iterrows():
    _col = component_to_column_name(_component)
    crash_cost_eval[_col] = estimated_crash_cost_for_component_row(
        crash_cost_eval, _units, is_pdo=is_pdo_crash
    )

_total_col = component_to_column_name("TotalComp.")
crash_cost_eval["estimated_total_comp_cost_2025usd"] = crash_cost_eval[_total_col]

UNIT_COST_MATRIX

,PDOVehicle,MAIS0,MAIS1,MAIS2,MAIS3,MAIS4,MAIS5,Fatal
Medical,0.00,0.00,2784.60,16718.94,87374.70,237668.76,457668.54,21784.14
EMS,90.72,50.40,175.14,345.24,612.36,1229.76,1258.74,1335.60
MarketProd.,0.00,0.00,2916.90,29100.96,116822.16,289677.78,385857.36,1273822.20
HouseholdProd.,89.46,69.30,1068.48,11327.40,49141.26,146767.32,161136.36,462606.48
InsuranceAdmin.,658.98,283.50,2787.12,10357.20,36159.48,45971.10,47982.06,45668.70
WorkplaceCosts,124.74,95.76,70.56,526.68,4082.40,8917.02,9820.44,17122.14
LegalCosts,0.00,0.00,932.40,7866.18,34919.64,92986.74,138615.12,173911.50
Congestion,3264.66,2191.14,2158.38,2215.08,2255.40,2285.64,2339.82,8987.58
Prop.Damage,5740.56,3344.04,17313.66,17251.92,31997.70,25911.90,29274.84,19133.10
Total Economic,9970.38,6034.14,30207.24,95710.86,363365.10,851416.02,1233953.28,2024371.44


In [9]:
# Spatial join: assign each crash to 2024 TIGER/Line polygons (Maryland only) for county, tract, and place.
# Uses NHGIS shape extract (`nhgis0008_shape.zip`). IDs (`GISJOIN`, `GEOID`, …) match NHGIS CSV tabular data;
# geometries stay on disk in `tl2024_shapes_cache` for maps (same paths geopandas reads below).

import io
import zipfile

import geopandas as gpd
import numpy as np

NHGIS_SHAPE_ZIP = DATA_DIR / "census" / "nhgis0008_shape.zip"
TL_SHAPE_CACHE = DATA_DIR / "census" / "tl2024_shapes_cache"
MD_STATEFP_WHERE = "STATEFP = '24'"

TL_INNER_ZIPS = {
    "county": "nhgis0008_shape/nhgis0008_shapefile_tl2024_us_county_2024.zip",
    "tract": "nhgis0008_shape/nhgis0008_shapefile_tl2024_us_tract_2024.zip",
    "place": "nhgis0008_shape/nhgis0008_shapefile_tl2024_us_place_2024.zip",
}
TL_SHP_NAMES = {
    "county": "US_county_2024.shp",
    "tract": "US_tract_2024.shp",
    "place": "US_place_2024.shp",
}

COUNTY_ID_COLS = ["GISJOIN", "GEOID", "GEOIDFQ", "STATEFP", "COUNTYFP", "NAME", "NAMELSAD"]
TRACT_ID_COLS = ["GISJOIN", "GEOID", "GEOIDFQ", "STATEFP", "COUNTYFP", "TRACTCE", "NAMELSAD"]
PLACE_ID_COLS = ["GISJOIN", "GEOID", "GEOIDFQ", "STATEFP", "PLACEFP", "NAME", "NAMELSAD"]


def ensure_tl2024_shp(layer: str) -> Path:
    """One-time unzip of nested NHGIS archive; return path to `.shp` for geopandas."""
    cache_dir = TL_SHAPE_CACHE / layer
    marker = cache_dir / ".extracted"
    if not marker.exists():
        cache_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(NHGIS_SHAPE_ZIP) as outer:
            blob = outer.read(TL_INNER_ZIPS[layer])
        with zipfile.ZipFile(io.BytesIO(blob)) as inner:
            inner.extractall(cache_dir)
        marker.write_text("ok", encoding="ascii")
    return cache_dir / TL_SHP_NAMES[layer]


def spatial_join_polygon_ids(
    target: pd.DataFrame,
    points: gpd.GeoDataFrame,
    polys: gpd.GeoDataFrame,
    id_cols: list[str],
    prefix: str,
) -> pd.DataFrame:
    """Left spatial join `within`; first match only if a point falls in multiple polygons."""
    right = polys[[c for c in id_cols if c in polys.columns] + ["geometry"]].copy()
    joined = gpd.sjoin(
        points[["_sjoin_row_id", "geometry"]],
        right,
        how="left",
        predicate="within",
    )
    joined = joined.drop(columns=["index_right"], errors="ignore")
    joined = joined.drop_duplicates(subset=["_sjoin_row_id"], keep="first").set_index(
        "_sjoin_row_id"
    )
    out = target.copy()
    for c in id_cols:
        if c in joined.columns:
            out[f"{c}_{prefix}"] = out["_sjoin_row_id"].map(joined[c])
    return out


crash_cost_eval["_sjoin_row_id"] = np.arange(len(crash_cost_eval), dtype=np.int64)

crash_pts = gpd.GeoDataFrame(
    {"_sjoin_row_id": crash_cost_eval["_sjoin_row_id"]},
    geometry=gpd.points_from_xy(
        crash_cost_eval["Longitude"], crash_cost_eval["Latitude"]
    ),
    crs="EPSG:4326",
)

md_county = gpd.read_file(ensure_tl2024_shp("county"), where=MD_STATEFP_WHERE).to_crs(
    4326
)
md_tract = gpd.read_file(ensure_tl2024_shp("tract"), where=MD_STATEFP_WHERE).to_crs(4326)
md_place = gpd.read_file(ensure_tl2024_shp("place"), where=MD_STATEFP_WHERE).to_crs(4326)

crash_cost_eval = spatial_join_polygon_ids(
    crash_cost_eval, crash_pts, md_county, COUNTY_ID_COLS, "county"
)
crash_cost_eval = spatial_join_polygon_ids(
    crash_cost_eval, crash_pts, md_tract, TRACT_ID_COLS, "tract"
)
crash_cost_eval = spatial_join_polygon_ids(
    crash_cost_eval, crash_pts, md_place, PLACE_ID_COLS, "place"
)

crash_cost_eval = crash_cost_eval.drop(columns=["_sjoin_row_id"])

(
    "county_polys",
    len(md_county),
    "tract_polys",
    len(md_tract),
    "place_polys",
    len(md_place),
    "crash_cost_eval",
    crash_cost_eval.shape,
)

('county_polys',
 24,
 'tract_polys',
 1463,
 'place_polys',
 536,
 'crash_cost_eval',
 (210860, 64))

In [10]:
CRASH_COST_EVAL_CSV = DATA_DIR / "crash_cost_eval.csv"
crash_cost_eval.to_csv(CRASH_COST_EVAL_CSV, index=False)
CRASH_COST_EVAL_CSV.resolve()

WindowsPath('C:/Users/Matt/OneDrive/personal/research/Perfect-Numbers/2026/crash_costs/data/crash_cost_eval.csv')

## Geographic summaries (census + crashes)

Run [`geographic_summaries.py`](../geographic_summaries.py) from the **`crash_costs`** folder (or execute the next cell) **after** exporting `crash_cost_eval.csv` **with** `GISJOIN_*` columns from the spatial-join step. Outputs: `data/geo_summaries/` — see [`data/geo_summaries/README.md`](../data/geo_summaries/README.md).

In [12]:
# Writes data/geo_summaries/*.csv (requires GISJOIN_* on crash_cost_eval.csv).
import subprocess
import sys

subprocess.run(
    [sys.executable, str(NOTEBOOK_DIR / "geographic_summaries.py")],
    cwd=NOTEBOOK_DIR,
    check=True,
)

CompletedProcess(args=['c:\\Users\\Matt\\AppData\\Local\\Programs\\Python\\Python313\\python.exe', 'c:\\Users\\Matt\\OneDrive\\personal\\research\\Perfect-Numbers\\2026\\crash_costs\\geographic_summaries.py'], returncode=0)